# M3GNet XAS pipeline showcase

This notebook displays the M3GNet encoder and XAS heads, then trains and evaluates the pipeline.

## Prerequisites

From the repository root, run:

```bash
bash tutorial_omnixas/download_omnixas_raw_data.sh
export OMNIXAS_DATA_ROOT="$HOME/OmniXAS_data"
```

The script downloads and extracts FEFF data by default. VASP download is not needed for this FEFF pipeline.

The shell script requires `curl`, `md5sum`, and `tar`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py').is_file():
    REPO_ROOT = REPO_ROOT.parent
SCRIPT = REPO_ROOT / 'tutorial_omnixas' / 'train_m3gnet_xas_pipeline.py'
from omnixas.model.m3gnet_xas import HEAD_HIDDEN_DIMS, FEATURE_SCALE, SPECTRUM_DIM, M3GNetXAS, XASSpectralHead

model = M3GNetXAS()
head = XASSpectralHead()
print(model)
print(f"M3GNetXAS parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"XASSpectralHead parameters: {sum(p.numel() for p in head.parameters()):,}")
print({'head_hidden_dims': HEAD_HIDDEN_DIMS, 'feature_scale': FEATURE_SCALE, 'target_dim': SPECTRUM_DIM})

In [ ]:
RUN_NAME = 'm3gnet_xas_seed42'
OUTPUT_ROOT = REPO_ROOT / 'output/training/m3gnet_xas_pipeline'
RUN_DIR = OUTPUT_ROOT / RUN_NAME

## Train the pipeline

The training script checks the input data before training. The command can take a long time.

In [ ]:
import runpy

script_args = [
    str(SCRIPT),
    '--output-root',
    str(OUTPUT_ROOT),
    '--run-name',
    RUN_NAME,
    '--precision',
    'bf16-mixed',
    '--encoder-rows-per-element',
    '256',
    '--encoder-eval-batch-size',
    '512',
    '--num-workers',
    '16',
    '--batch-size',
    '4096',
]
previous_argv = sys.argv
sys.argv = script_args
try:
    runpy.run_path(str(SCRIPT), run_name='__main__')
finally:
    sys.argv = previous_argv

## Evaluate the trained pipeline

The script writes the four metric CSV files `universal_validation.csv`,
`universal_test.csv`, `tuned_validation.csv`, and `tuned_test.csv` into the run directory.

In [ ]:
subprocess.run([
    sys.executable,
    str(SCRIPT),
    '--output-root',
    str(OUTPUT_ROOT),
    '--run-name',
    RUN_NAME,
    '--evaluate',
], cwd=REPO_ROOT, check=True)

## Evaluation results

These cells read the metric CSVs and head checkpoints from the run directory.
They compare the UniversalXAS head and the tuned heads on the validation and test splits.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

universal_val = pd.read_csv(RUN_DIR / "universal_validation.csv")
universal_test = pd.read_csv(RUN_DIR / "universal_test.csv")
tuned_val = pd.read_csv(RUN_DIR / "tuned_validation.csv")
tuned_test = pd.read_csv(RUN_DIR / "tuned_test.csv")


elements = [task.removesuffix("_FEFF") for task in universal_val["dataset"]]
per_element_table = pd.DataFrame(
    {
        "UniversalXAS val eta": universal_val["val_eta"].to_numpy(),
        "UniversalXAS val median MSE": universal_val["val_median_mse"].to_numpy(),
        "Tuned val eta": tuned_val["eta"].to_numpy(),
        "Tuned val median MSE": tuned_val["median_mse"].to_numpy(),
        "UniversalXAS test eta": universal_test["test_eta"].to_numpy(),
        "UniversalXAS test median MSE": universal_test["test_median_mse"].to_numpy(),
        "Tuned test eta": tuned_test["eta"].to_numpy(),
        "Tuned test median MSE": tuned_test["median_mse"].to_numpy(),
    },
    index=pd.Index(elements, name="element"),
)
per_element_table = per_element_table.round(
    {col: 3 if "eta" in col else 6 for col in per_element_table.columns}
)
per_element_table

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


pairs = [
    (RUN_DIR / "universal_validation.csv", "val_eta",
     RUN_DIR / "tuned_validation.csv", "eta",
     "Validation eta by element"),
    (RUN_DIR / "universal_test.csv", "test_eta",
     RUN_DIR / "tuned_test.csv", "eta",
     "Test eta by element"),
]


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (u_path, u_col, t_path, t_col, title) in zip(axes, pairs):
    universal = pd.read_csv(u_path)
    tuned = pd.read_csv(t_path)
    x = np.arange(len(universal))
    width = 0.35
    ax.bar(x - width / 2, universal[u_col].to_numpy(), width, label="UniversalXAS")
    ax.bar(x + width / 2, tuned[t_col].to_numpy(), width, label="Tuned UniversalXAS")
    ax.set_xticks(x, [task.removesuffix("_FEFF") for task in universal["dataset"]])
    ax.set_ylabel("eta")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()

In [ ]:
import pandas as pd


specs = [
    ("UniversalXAS validation", RUN_DIR / "universal_validation.csv",
     "val_eta", "val_mse", "val_baseline_median_mse"),
    ("UniversalXAS test", RUN_DIR / "universal_test.csv",
     "test_eta", "test_mse", "test_baseline_median_mse"),
    ("Tuned validation", RUN_DIR / "tuned_validation.csv",
     "eta", "mse", "baseline_median_mse"),
    ("Tuned test", RUN_DIR / "tuned_test.csv",
     "eta", "mse", "baseline_median_mse"),
]


rows = {}
for name, path, eta_col, mse_col, base_col in specs:
    frame = pd.read_csv(path)
    rows[name] = [
        frame[eta_col].mean(),
        frame[eta_col].median(),
        frame[mse_col].mean(),
        frame[base_col].mean(),
    ]


aggregate_table = pd.DataFrame(rows, index=[
    "mean eta", "median eta", "mean MSE", "mean baseline median MSE"
]).T
aggregate_table = aggregate_table.round(
    {"mean eta": 3, "mean MSE": 6, "mean baseline median MSE": 6}
)


delta = (
    pd.read_csv(RUN_DIR / "tuned_test.csv")["eta"].to_numpy()
    - pd.read_csv(RUN_DIR / "universal_test.csv")["test_eta"].to_numpy()
).mean()
print(f"Mean per-element delta (Tuned minus UniversalXAS) on test: {delta:.3f}")
aggregate_table

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from omnixas.model.m3gnet_xas import XASSpectralHead


def load_head(path):
    head = XASSpectralHead()
    state = torch.load(path, map_location="cpu")["state_dict"]
    head.load_state_dict(state, strict=True)
    head.eval()
    return head


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tasks = ["Ti_FEFF", "Mn_FEFF", "Fe_FEFF", "Cu_FEFF"]
universal_head = load_head(RUN_DIR / "heads" / "universalXAS" / "best.pt").to(device)


fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for ax, task in zip(axes.ravel(), tasks):
    X = np.atleast_2d(np.loadtxt(RUN_DIR / f"features/{task}_test_X.txt", dtype=np.float32))
    y = np.atleast_2d(np.loadtxt(RUN_DIR / f"features/{task}_test_y.txt", dtype=np.float32))
    tuned_head = load_head(RUN_DIR / "heads" / "tunedUniversalXAS" / task / "best.pt").to(device)
    with torch.inference_mode():
        x0 = torch.tensor(X[:1], dtype=torch.float32, device=device)
        u_pred = universal_head(x0).cpu().numpy().ravel()
        t_pred = tuned_head(x0).cpu().numpy().ravel()
    grid = np.arange(141)
    ax.plot(grid, y[0], label="target")
    ax.plot(grid, u_pred, label="UniversalXAS")
    ax.plot(grid, t_pred, label="Tuned UniversalXAS")
    ax.set_title(task.removesuffix("_FEFF"))
    ax.set_xlabel("energy grid index")
    ax.set_ylabel("intensity")
    ax.legend()
plt.tight_layout()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


version_dirs = sorted(
    (RUN_DIR / "encoder_logs").glob("version_*"),
    key=lambda p: int(p.name.split("_")[1]),
)
if not version_dirs:
    raise FileNotFoundError(f"No encoder_logs/version_* directory under {RUN_DIR}")
frames = [pd.read_csv(d / "metrics.csv") for d in version_dirs if (d / "metrics.csv").is_file()]
if not frames:
    raise FileNotFoundError(f"No metrics.csv under {RUN_DIR / 'encoder_logs'}")
metrics = pd.concat(frames, ignore_index=True)


def find_metric(frame, *keywords):
    for col in frame.columns:
        if all(k in col.lower() for k in keywords):
            return col
    return None


train_loss_col = find_metric(metrics, "train", "loss")
if train_loss_col is None and "loss" in [c.lower() for c in metrics.columns]:
    train_loss_col = "loss"
val_loss_col = find_metric(metrics, "val", "loss")
rel_mse_col = find_metric(metrics, "balanced")
if not (train_loss_col and val_loss_col and rel_mse_col):
    raise ValueError(
        f"metrics.csv lacks expected metric columns (train loss, val loss, balanced rel MSE). "
        f"Columns: {list(metrics.columns)}"
    )

counts = {c: int(metrics[c].notna().sum()) for c in (train_loss_col, val_loss_col, rel_mse_col)}
if any(n == 0 for n in counts.values()):
    raise ValueError(f"No numeric values in metric columns {counts} in {version_dirs}")


def series(col):
    part = metrics.dropna(subset=[col])
    x = part["epoch"] if "epoch" in part.columns else part.index
    return x, part[col]


train_x, train_y = series(train_loss_col)
val_x, val_y = series(val_loss_col)
rel_x, rel_y = series(rel_mse_col)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_x, train_y, label=train_loss_col)
axes[0].plot(val_x, val_y, label=val_loss_col)
axes[0].set_xlabel("epoch")
axes[0].set_title("Encoder loss")
axes[0].legend()
axes[1].plot(rel_x, rel_y)
axes[1].set_xlabel("epoch")
axes[1].set_title("Validation balanced relative MSE")
plt.tight_layout()
